# nb01 — QC and Structure Inspection
Load CITE-seq and Multiome h5ad files. Understand what's already been done — no transforms applied here, exploration only.

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q scanpy anndata
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics-relationship-modeling')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## Imports and config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = BASE_PATH / 'data' / 'benchmark'
CITE_H5AD     = DATA_DIR / 'GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad'
MULTIOME_H5AD = DATA_DIR / 'GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad'

RESULTS_DIR = BASE_PATH / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('CITE exists:', CITE_H5AD.exists())
print('Multiome exists:', MULTIOME_H5AD.exists())

## Load CITE-seq

In [ ]:
cite = sc.read_h5ad(CITE_H5AD)
print(cite)

### CITE-seq — obs, var, obsm, uns, layers

In [ ]:
print('--- obs columns ---')
print(cite.obs.columns.tolist())
print()
print('--- obs head ---')
display(cite.obs.head())

In [ ]:
print('--- var columns ---')
print(cite.var.columns.tolist())
print()
display(cite.var.head())

In [ ]:
print('--- obsm keys ---')
print(list(cite.obsm.keys()))
for k in cite.obsm.keys():
    print(f'  {k}: shape {cite.obsm[k].shape}')

In [ ]:
print('--- layers ---')
print(list(cite.layers.keys()))
print()
print('--- uns keys ---')
print(list(cite.uns.keys()))

### CITE-seq — is X raw counts, normalized, or log-transformed?

In [ ]:
import scipy.sparse as sp

X = cite.X
if sp.issparse(X):
    X_sample = X[:1000].toarray()
else:
    X_sample = np.asarray(X[:1000])

print('X dtype:', X.dtype)
print('X min/max:', X_sample.min(), X_sample.max())
print('X mean:', X_sample.mean())
print('Any negative values:', (X_sample < 0).any())
print('Any non-integer values (first 1000 cells):', not np.allclose(X_sample, np.round(X_sample)))

### CITE-seq — donor / cell type / batch columns (if present)

In [ ]:
for col in ['donor', 'donor_id', 'cell_type', 'celltype', 'batch', 'site', 'Site']:
    if col in cite.obs.columns:
        print(f'--- {col} ---')
        print(cite.obs[col].value_counts())
        print()

## Load Multiome

In [ ]:
multiome = sc.read_h5ad(MULTIOME_H5AD)
print(multiome)

### Multiome — obs, var, obsm, uns, layers

In [ ]:
print('--- obs columns ---')
print(multiome.obs.columns.tolist())
display(multiome.obs.head())

In [ ]:
print('--- var columns ---')
print(multiome.var.columns.tolist())
display(multiome.var.head())

In [ ]:
print('--- obsm keys ---')
print(list(multiome.obsm.keys()))
for k in multiome.obsm.keys():
    print(f'  {k}: shape {multiome.obsm[k].shape}')

In [ ]:
print('--- layers ---')
print(list(multiome.layers.keys()))
print()
print('--- uns keys ---')
print(list(multiome.uns.keys()))

### Multiome — is X raw, normalized, or log-transformed?

In [ ]:
X2 = multiome.X
if sp.issparse(X2):
    X2_sample = X2[:1000].toarray()
else:
    X2_sample = np.asarray(X2[:1000])

print('X dtype:', X2.dtype)
print('X min/max:', X2_sample.min(), X2_sample.max())
print('X mean:', X2_sample.mean())
print('Any negative values:', (X2_sample < 0).any())

### Multiome — donor / cell type / batch columns (if present)

In [ ]:
for col in ['donor', 'donor_id', 'cell_type', 'celltype', 'batch', 'site', 'Site']:
    if col in multiome.obs.columns:
        print(f'--- {col} ---')
        print(multiome.obs[col].value_counts())
        print()

## QC summary plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

total_counts_cite = np.asarray(X.sum(axis=1)).flatten() if sp.issparse(X) else X.sum(axis=1)
sns.histplot(total_counts_cite, bins=50, ax=axes[0])
axes[0].set_title('CITE-seq: total counts per cell')

total_counts_multi = np.asarray(X2.sum(axis=1)).flatten() if sp.issparse(X2) else X2.sum(axis=1)
sns.histplot(total_counts_multi, bins=50, ax=axes[1])
axes[1].set_title('Multiome: total counts per cell')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb01_total_counts.png', dpi=150)
plt.show()

## Barcode overlap check
Confirms whether CITE-seq and Multiome share the same donors/cells, or are fully separate cell populations (expected for this benchmark).

In [ ]:
common_barcodes = set(cite.obs_names) & set(multiome.obs_names)
print(f'CITE-seq cells: {cite.n_obs:,}')
print(f'Multiome cells: {multiome.n_obs:,}')
print(f'Common barcodes: {len(common_barcodes):,}')

## Summary findings
Fill in after running the cells above:
- X representation (raw/normalized/log): 
- Cell type annotations present: 
- Donor/batch columns: 
- Protein (ADT) location: 
- ATAC location: 
- Any cells shared between CITE-seq and Multiome: 